In [1]:
# ============================================================
# PASS 4 — INSTALL BASELINE DEPENDENCIES
# ============================================================

# Do NOT upgrade the whole scientific Python stack on Kaggle.
# Keep Kaggle's existing NumPy/Pandas/scikit-learn versions.
# Only install the packages required by Pass 4.

!pip -q install lightgbm xgboost pyyaml

# cesnet-datazoo is normally already available in the environment.
# Install it only if the import is unavailable.

try:
    import cesnet_datazoo
    print("cesnet-datazoo: already installed")
except ImportError:
    print("cesnet-datazoo not found; installing...")
    !pip -q install cesnet-datazoo

print("Pass 4 dependencies ready.")

cesnet-datazoo not found; installing...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 45.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sigstore-models 0.0.6 requires pydantic>=2.12, but you have pydantic 2.11.10 which is incompatible.
google-genai 2.12.1 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 2.11.10 which is incompatible.
google-adk 2.7.1 requires pydantic<3,>=2.12, but you have pydantic 2.11.10 which is incompatible.
Pass 4 dependencies ready.


In [2]:
# ============================================================
# PASS 4 — IMPORTS, REPRODUCIBILITY, PATHS, CONFIGURATION
# ============================================================

import os
import gc
import json
import time
import math
import warnings
import hashlib
import shutil
import subprocess
import importlib.metadata as importlib_metadata
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import yaml

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Global reproducibility
# ------------------------------------------------------------

SEED = 20261003
np.random.seed(SEED)

# ------------------------------------------------------------
# Experiment configuration
# ------------------------------------------------------------

DATASET_SIZE = "M"

# The neural Pass 3 used the complete Jun 2024-Mar 2025
# training period (27,727,267 flows).
#
# For classical tree baselines we use a deterministic uniform
# 4-million-flow sample to keep memory/training requirements
# manageable while preserving the complete temporal period.
TRAIN_BASELINE_SAMPLES = 4_000_000

# April validation sample.
VAL_SAMPLES = 500_000

# HDF5 reading chunk.
READ_CHUNK = 100_000

# PPI shape.
PPI_PACKETS = 30
PPI_CHANNELS = 3

# Output directory.
RESULT_DIR = Path(
    "/kaggle/working/pass4_results"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Locate the exact attached CESNET-QUICEXT-25-M database.
#
# Your current Kaggle input is:
#
# /kaggle/input/datasets/yahiaa1/
# cesnet-quixext-25-m/CESNET-QUICEXT-25/
# M/CESNET-QUICEXT-25-M.h5
#
# We discover it rather than hard-coding the Kaggle username
# or dataset slug.
# ------------------------------------------------------------

INPUT_ROOT = Path("/kaggle/input")

if not INPUT_ROOT.exists():
    raise FileNotFoundError(
        "/kaggle/input does not exist."
    )

matches = list(
    INPUT_ROOT.rglob(
        "CESNET-QUICEXT-25-M.h5"
    )
)

if not matches:
    raise FileNotFoundError(
        "CESNET-QUICEXT-25-M.h5 was not found "
        "under /kaggle/input.\n\n"
        "Attach the CESNET-QUICEXT-25-M dataset "
        "through Kaggle -> Input -> Add Input."
    )

if len(matches) > 1:
    print(
        "WARNING: multiple M databases found:"
    )

    for i, path in enumerate(matches):
        print(
            f"  [{i}] {path}"
        )

# Use the first exact database match.
DB_PATH = matches[0]

# Directory:
#   .../CESNET-QUICEXT-25/M/
DATASET_M_ROOT = DB_PATH.parent

# Directory:
#   .../CESNET-QUICEXT-25/
DATA_ROOT = DATASET_M_ROOT.parent

print("=" * 80)
print("DATASET PATH")
print("=" * 80)

print(
    "DATA_ROOT      :",
    DATA_ROOT
)

print(
    "DATASET_M_ROOT :",
    DATASET_M_ROOT
)

print(
    "DB_PATH        :",
    DB_PATH
)

print(
    "DB exists      :",
    DB_PATH.exists()
)

print(
    "Database size  :",
    f"{DB_PATH.stat().st_size / (1024**3):.2f} GB"
)

assert DB_PATH.exists()

# ------------------------------------------------------------
# Package versions.
# ------------------------------------------------------------

PACKAGE_NAMES = [
    "cesnet-datazoo",
    "numpy",
    "pandas",
    "scikit-learn",
    "lightgbm",
    "xgboost",
    "pyyaml",
    "joblib",
]

VERSIONS = {}

for pkg in PACKAGE_NAMES:

    try:
        VERSIONS[pkg] = (
            importlib_metadata.version(pkg)
        )

    except Exception:
        VERSIONS[pkg] = "unknown"

print(
    "\n" + "=" * 80
)

print(
    "PACKAGE VERSIONS"
)

print(
    "=" * 80
)

for k, v in VERSIONS.items():
    print(
        f"{k:20s}: {v}"
    )

# ------------------------------------------------------------
# GPU visibility.
# ------------------------------------------------------------

GPU_VISIBLE = False

try:

    result = subprocess.run(
        ["nvidia-smi", "-L"],
        capture_output=True,
        text=True,
        timeout=10,
    )

    GPU_VISIBLE = (
        result.returncode == 0
        and bool(result.stdout.strip())
    )

except Exception:
    GPU_VISIBLE = False

print(
    "\nNVIDIA GPU visible:",
    GPU_VISIBLE
)

if GPU_VISIBLE:

    try:

        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total",
                "--format=csv,noheader",
            ],
            capture_output=True,
            text=True,
            timeout=10,
        )

        print(
            result.stdout.strip()
        )

    except Exception:
        pass

# ------------------------------------------------------------
# Initial manifest.
# ------------------------------------------------------------

INITIAL_MANIFEST = {
    "pass": "4",
    "seed": SEED,
    "dataset": "CESNET-QUICEXT-25",
    "dataset_size": DATASET_SIZE,
    "database_path": str(DB_PATH),
    "train_baseline_samples": (
        TRAIN_BASELINE_SAMPLES
    ),
    "validation_samples": VAL_SAMPLES,
    "ppi_packets": PPI_PACKETS,
    "ppi_channels": [
        "IPT",
        "DIR",
        "SIZE",
    ],
    "result_dir": str(
        RESULT_DIR
    ),
    "versions": VERSIONS,
    "gpu_visible": GPU_VISIBLE,
}

with open(
    RESULT_DIR
    / "pass4_initial_manifest.json",
    "w",
) as f:

    json.dump(
        INITIAL_MANIFEST,
        f,
        indent=2,
    )

print(
    "\nCell 1 complete."
)

DATASET PATH
DATA_ROOT      : /kaggle/input/datasets/yahiaa1/cesnet-quixext-25-m/CESNET-QUICEXT-25
DATASET_M_ROOT : /kaggle/input/datasets/yahiaa1/cesnet-quixext-25-m/CESNET-QUICEXT-25/M
DB_PATH        : /kaggle/input/datasets/yahiaa1/cesnet-quixext-25-m/CESNET-QUICEXT-25/M/CESNET-QUICEXT-25-M.h5
DB exists      : True
Database size  : 16.44 GB

PACKAGE VERSIONS
cesnet-datazoo      : 0.2.0
numpy               : 2.1.3
pandas              : 2.3.3
scikit-learn        : 1.6.1
lightgbm            : 4.6.0
xgboost             : 3.4.1
pyyaml              : 6.0.3
joblib              : 1.6.0

NVIDIA GPU visible: True
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

Cell 1 complete.


In [3]:
# ============================================================
# PASS 4 — INITIALIZE CESNET DATAZOO
# + EXACT 50-CLASS LABEL MAPPING
# ============================================================

from cesnet_datazoo.datasets import (
    CESNET_QUICEXT_25,
)

from cesnet_datazoo.config import (
    DatasetConfig,
    AppSelection,
)

# ------------------------------------------------------------
# Initialize DataZoo.
# ------------------------------------------------------------

dataset = CESNET_QUICEXT_25(
    str(DATA_ROOT),
    size=DATASET_SIZE,
)

dataset_config = DatasetConfig(
    dataset=dataset,
    apps_selection=AppSelection.ALL_KNOWN,
    train_period_name="M-2025-1",
    test_period_name="M-2025-2",
)

dataset.set_dataset_config_and_initialize(
    dataset_config
)

print("=" * 80)
print("DATASET INITIALIZED")
print("=" * 80)

print(
    "Dataset:",
    dataset.name
)

print(
    "Database:",
    dataset.database_path
)

# ------------------------------------------------------------
# IMPORTANT:
#
# Do not use:
#
#     dataset.get_known_apps()
#
# because that helper is not guaranteed to exist in the
# installed DataZoo version.
#
# The attached dataset already contains:
#
#   M/train-data/.../known_apps.yaml
#
# We use that exact metadata.
# ------------------------------------------------------------

KNOWN_APPS_CANDIDATES = list(
    DATASET_M_ROOT.glob(
        "train-data/*/fold_*/known_apps.yaml"
    )
)

if not KNOWN_APPS_CANDIDATES:

    raise FileNotFoundError(
        "Could not find known_apps.yaml under:\n"
        f"{DATASET_M_ROOT / 'train-data'}"
    )

if len(KNOWN_APPS_CANDIDATES) > 1:

    print(
        "\nMultiple known_apps.yaml files found:"
    )

    for i, path in enumerate(
        KNOWN_APPS_CANDIDATES
    ):
        print(
            f"  [{i}] {path}"
        )

KNOWN_APPS_PATH = (
    KNOWN_APPS_CANDIDATES[0]
)

print(
    "\nKnown-app metadata:"
)

print(
    KNOWN_APPS_PATH
)

# ------------------------------------------------------------
# Load YAML.
# ------------------------------------------------------------

with open(
    KNOWN_APPS_PATH,
    "r",
) as f:

    known_apps_data = (
        yaml.safe_load(f)
    )

print(
    "\nYAML type:",
    type(known_apps_data)
)

# ------------------------------------------------------------
# Extract application names robustly.
# ------------------------------------------------------------

KNOWN_APPS = None

# Case 1:
#   known_apps:
#     - app1
#     - app2
if isinstance(
    known_apps_data,
    dict,
):

    for key in [
        "known_apps",
        "apps",
        "applications",
        "classes",
    ]:

        if key not in known_apps_data:
            continue

        value = (
            known_apps_data[key]
        )

        if isinstance(
            value,
            list,
        ):

            KNOWN_APPS = [
                str(v)
                for v in value
            ]

            break

        if isinstance(
            value,
            dict,
        ):

            KNOWN_APPS = [
                str(v)
                for v in value.values()
                if isinstance(
                    v,
                    (
                        str,
                        int,
                        float,
                    ),
                )
            ]

            break

# Case 2:
# YAML itself is a list.
elif isinstance(
    known_apps_data,
    list,
):

    # Handle either:
    #   ["google", "youtube", ...]
    #
    # or:
    #   [{"name": "google"}, ...]
    if all(
        isinstance(
            v,
            str,
        )
        for v in known_apps_data
    ):

        KNOWN_APPS = [
            str(v)
            for v in known_apps_data
        ]

    elif all(
        isinstance(
            v,
            dict,
        )
        for v in known_apps_data
    ):

        possible_name_keys = [
            "name",
            "app",
            "application",
            "label",
        ]

        extracted = []

        for item in known_apps_data:

            found = None

            for key in possible_name_keys:

                if key in item:

                    found = item[key]
                    break

            if found is not None:
                extracted.append(
                    str(found)
                )

        if extracted:
            KNOWN_APPS = extracted

# ------------------------------------------------------------
# Last-resort dictionary handling:
#
# Some metadata files can be mappings such as:
#
#   0: google-www
#   1: youtube
#   ...
# ------------------------------------------------------------

if KNOWN_APPS is None:

    if isinstance(
        known_apps_data,
        dict,
    ):

        scalar_values = [
            str(v)
            for v in known_apps_data.values()
            if isinstance(
                v,
                str,
            )
        ]

        if scalar_values:
            KNOWN_APPS = (
                scalar_values
            )

# ------------------------------------------------------------
# Validate.
# ------------------------------------------------------------

if KNOWN_APPS is None:

    print(
        "\nCould not automatically parse "
        "known_apps.yaml."
    )

    print(
        "\nContents of YAML:"
    )

    print(
        repr(known_apps_data)
    )

    raise ValueError(
        "Unable to extract known application labels "
        "from known_apps.yaml."
    )

# Remove duplicates while preserving order first.
KNOWN_APPS = list(
    dict.fromkeys(
        KNOWN_APPS
    )
)

# Canonical deterministic ordering.
KNOWN_APPS = sorted(
    KNOWN_APPS
)

N_CLASSES = len(
    KNOWN_APPS
)

print(
    "\nNumber of known applications:",
    N_CLASSES
)

if N_CLASSES != 50:

    print(
        "\nDetected classes:"
    )

    for i, name in enumerate(
        KNOWN_APPS
    ):
        print(
            f"{i:3d} -> {name}"
        )

    raise RuntimeError(
        f"Expected 50 known classes, "
        f"but detected {N_CLASSES}."
    )

# ------------------------------------------------------------
# Build canonical ID mapping.
# ------------------------------------------------------------

CLASS_TO_ID = {
    name: i
    for i, name in enumerate(
        KNOWN_APPS
    )
}

ID_TO_CLASS = {
    i: name
    for name, i in CLASS_TO_ID.items()
}

print(
    "\n" + "=" * 80
)

print(
    "50-CLASS LABEL MAPPING"
)

print(
    "=" * 80
)

print(
    "\nFirst 10 classes:"
)

for i, name in enumerate(
    KNOWN_APPS[:10]
):
    print(
        f"{i:2d} -> {name}"
    )

print(
    "\nLast 10 classes:"
)

start_idx = (
    N_CLASSES - 10
)

for i, name in enumerate(
    KNOWN_APPS[-10:],
    start=start_idx,
):
    print(
        f"{i:2d} -> {name}"
    )

# ------------------------------------------------------------
# Save mapping.
# ------------------------------------------------------------

with open(
    RESULT_DIR
    / "label_mapping.json",
    "w",
) as f:

    json.dump(
        {
            "class_to_id": CLASS_TO_ID,
            "id_to_class": {
                str(k): v
                for k, v
                in ID_TO_CLASS.items()
            },
        },
        f,
        indent=2,
    )

# ------------------------------------------------------------
# Build temporal partitions.
#
# This deliberately keeps the same temporal protocol:
#
# TRAIN = June 2024 -> March 2025
# VAL   = April 2025
# TEST  = May 2025
#
# Test is identified here but NOT read.
# ------------------------------------------------------------

all_dates = sorted(
    {
        str(d)
        for period_dates
        in dataset.time_periods.values()
        for d in period_dates
    }
)

TRAIN_DATES = [
    d
    for d in all_dates
    if "20240601" <= d <= "20250331"
]

VAL_DATES = [
    d
    for d in all_dates
    if "20250401" <= d <= "20250430"
]

TEST_DATES = [
    d
    for d in all_dates
    if "20250501" <= d <= "20250531"
]

print(
    "\n" + "=" * 80
)

print(
    "TEMPORAL PARTITIONS"
)

print(
    "=" * 80
)

print(
    "TRAIN:",
    TRAIN_DATES[0],
    "->",
    TRAIN_DATES[-1],
    "| days:",
    len(TRAIN_DATES),
)

print(
    "VAL  :",
    VAL_DATES[0],
    "->",
    VAL_DATES[-1],
    "| days:",
    len(VAL_DATES),
)

print(
    "TEST :",
    TEST_DATES[0],
    "->",
    TEST_DATES[-1],
    "| days:",
    len(TEST_DATES),
)

assert (
    len(TRAIN_DATES) > 200
)

assert (
    len(VAL_DATES) >= 20
)

assert (
    len(TEST_DATES) >= 20
)

print(
    "\n50-class mapping and temporal partitions ready."
)

print(
    "TEST DATA HAS NOT BEEN READ."
)

OSError: [Errno 30] Read-only file system: '/kaggle/input/datasets/yahiaa1/cesnet-quixext-25-m/CESNET-QUICEXT-25/M/train-data/223c17fa79_420'

In [ ]:
# ============================================================
# PASS 4 — INITIALIZE CESNET DATAZOO
# ============================================================

from cesnet_datazoo.datasets import CESNET_QUICEXT_25
from cesnet_datazoo.config import DatasetConfig, AppSelection

dataset = CESNET_QUICEXT_25(
    str(DATA_ROOT),
    size=DATASET_SIZE,
)

dataset_config = DatasetConfig(
    dataset=dataset,
    apps_selection=AppSelection.ALL_KNOWN,
    train_period_name="M-2025-1",
    test_period_name="M-2025-2",
)

dataset.set_dataset_config_and_initialize(dataset_config)

KNOWN_APPS = sorted(dataset.get_known_apps())
N_CLASSES = len(KNOWN_APPS)

print("Dataset:", dataset.name)
print("Database:", dataset.database_path)
print("Number of known classes:", N_CLASSES)

print("\nFirst 10 classes:")
for name in KNOWN_APPS[:10]:
    print(" ", name)

print("\nLast 10 classes:")
for name in KNOWN_APPS[-10:]:
    print(" ", name)

assert N_CLASSES == 50, f"Expected 50 classes, found {N_CLASSES}"

# ------------------------------------------------------------
# Build exact date partitions independently of DataZoo's
# internal validation/test representation.
# ------------------------------------------------------------

all_dates = sorted(
    {
        str(d)
        for period_dates in dataset.time_periods.values()
        for d in period_dates
    }
)

TRAIN_DATES = [
    d for d in all_dates
    if "20240601" <= d <= "20250331"
]

VAL_DATES = [
    d for d in all_dates
    if "20250401" <= d <= "20250430"
]

TEST_DATES = [
    d for d in all_dates
    if "20250501" <= d <= "20250531"
]

print("\nTemporal partitions:")
print(
    "TRAIN:",
    TRAIN_DATES[0],
    "->",
    TRAIN_DATES[-1],
    "| days:",
    len(TRAIN_DATES),
)
print(
    "VAL  :",
    VAL_DATES[0],
    "->",
    VAL_DATES[-1],
    "| days:",
    len(VAL_DATES),
)
print(
    "TEST :",
    TEST_DATES[0],
    "->",
    TEST_DATES[-1],
    "| days:",
    len(TEST_DATES),
)

assert len(TRAIN_DATES) > 200
assert len(VAL_DATES) > 20
assert len(TEST_DATES) > 20

CLASS_TO_ID = {name: i for i, name in enumerate(KNOWN_APPS)}
ID_TO_CLASS = {i: name for name, i in CLASS_TO_ID.items()}

print("\n50-class label mapping created.")
print("TEST DATA HAS NOT BEEN READ.")

In [ ]:
# ============================================================
# PASS 4 — EXACT 101-FEATURE SCHEMA
# ============================================================

# ------------------------------------------------------------
# PPI columns
# ------------------------------------------------------------

IPT_COLUMNS = [f"IPT_{i}" for i in range(1, PPI_PACKETS + 1)]
DIR_COLUMNS = [f"DIR_{i}" for i in range(1, PPI_PACKETS + 1)]
SIZE_COLUMNS = [f"SIZE_{i}" for i in range(1, PPI_PACKETS + 1)]

# The current paper/model representation:
#   30 IPT + 30 DIR + 30 SIZE + 11 auxiliary = 101 features.

AUX_FEATURES = [
    "BYTES",
    "BYTES_REV",
    "PACKETS",
    "PACKETS_REV",
    "DURATION",
    "FLOW_ENDREASON_ACTIVE",
    "FLOW_ENDREASON_IDLE",
    "FLOW_ENDREASON_OTHER",
    "PPI_DURATION",
    "PPI_LEN",
    "PPI_ROUNDTRIPS",
]

FEATURE_COLUMNS = (
    IPT_COLUMNS
    + DIR_COLUMNS
    + SIZE_COLUMNS
    + AUX_FEATURES
)

N_FEATURES = len(FEATURE_COLUMNS)

assert N_FEATURES == 101, f"Expected 101 features, found {N_FEATURES}"

RAW_COLUMNS = [
    "PPI",
    *AUX_FEATURES,
    "APP",
]

print("PPI features :", 90)
print("Aux features :", len(AUX_FEATURES))
print("Total        :", N_FEATURES)

print("\nAuxiliary features:")
for i, feature in enumerate(AUX_FEATURES, 1):
    print(f"{i:2d}. {feature}")

print("\nRaw HDF columns:")
print(RAW_COLUMNS)

In [ ]:
# ============================================================
# PASS 4 — HDF5 SANITY CHECK
# ============================================================

import pandas as pd

store = pd.HDFStore(str(DB_PATH), mode="r")

try:
    hdf_keys = list(store.keys())

    flow_keys = [
        key for key in hdf_keys
        if key.startswith("/flows/D")
    ]

    print("Number of /flows/D... keys:", len(flow_keys))
    print("First keys:")
    for key in flow_keys[:5]:
        print(" ", key)

    assert flow_keys, "No /flows/DYYYYMMDD HDF5 tables found."

    sample_key = flow_keys[0]

    sample_df = store.select(
        sample_key,
        start=0,
        stop=3,
        columns=RAW_COLUMNS,
    )

    print("\nSample key:", sample_key)
    print("Shape:", sample_df.shape)
    print("Columns:", list(sample_df.columns))

    assert "PPI" in sample_df.columns
    assert "APP" in sample_df.columns

    first_ppi = np.asarray(sample_df["PPI"].iloc[0])

    print("\nFirst PPI raw shape:", first_ppi.shape)
    print("First PPI dtype:", first_ppi.dtype)

    print("\nFirst few PPI values:")
    print(first_ppi[:5])

finally:
    store.close()

print("\nHDF5 sanity check complete.")

In [ ]:
# ============================================================
# PASS 4 — RAW HDF5 READER
# ============================================================

def normalize_app_label(value):
    """
    Convert HDF5 APP values to the canonical string representation.
    """
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    return str(value)


def parse_single_ppi(value):
    """
    Convert one raw PPI object into shape (30, 3).

    Expected channel order for this project:
        column 0 = IPT
        column 1 = DIR
        column 2 = SIZE
    """
    arr = np.asarray(value, dtype=np.float32)

    if arr.ndim != 2:
        raise ValueError(
            f"Unexpected PPI ndim={arr.ndim}, shape={arr.shape}"
        )

    # Normal expected shape.
    if arr.shape[0] >= 30 and arr.shape[1] >= 3:
        arr = arr[:30, :3]

    # Defensive handling if stored transposed.
    elif arr.shape[0] >= 3 and arr.shape[1] >= 30:
        arr = arr[:3, :30].T

    else:
        raise ValueError(
            f"Cannot convert PPI shape {arr.shape} to (30,3)"
        )

    if arr.shape != (30, 3):
        raise ValueError(
            f"Final PPI shape is {arr.shape}, expected (30,3)"
        )

    return arr


def raw_dataframe_to_numpy(raw_df):
    """
    Convert a raw HDF5 chunk to:
        X shape = (N, 101)
        y shape = (N,)
    """

    ppi_arrays = [
        parse_single_ppi(value)
        for value in raw_df["PPI"].to_numpy()
    ]

    ppi = np.stack(ppi_arrays, axis=0).astype(
        np.float32,
        copy=False,
    )

    # Convert from (N, 30, 3) into:
    # IPT_1...IPT_30,
    # DIR_1...DIR_30,
    # SIZE_1...SIZE_30
    ppi_flat = np.concatenate(
        [
            ppi[:, :, 0],
            ppi[:, :, 1],
            ppi[:, :, 2],
        ],
        axis=1,
    )

    aux = raw_df[AUX_FEATURES].to_numpy(
        dtype=np.float32,
        copy=True,
    )

    X = np.concatenate(
        [ppi_flat, aux],
        axis=1,
    ).astype(np.float32, copy=False)

    labels = np.array(
        [
            normalize_app_label(v)
            for v in raw_df["APP"].to_numpy()
        ],
        dtype=object,
    )

    y = np.fromiter(
        (CLASS_TO_ID[label] for label in labels),
        dtype=np.int16,
        count=len(labels),
    )

    assert X.shape[1] == 101

    return X, y


def iter_period_chunks(dates, read_chunk=READ_CHUNK):
    """
    Stream raw known-class flows from HDF5 by date/chunk.
    """

    store = pd.HDFStore(str(DB_PATH), mode="r")

    try:
        available_keys = set(store.keys())

        for date_str in dates:

            key = f"/flows/D{date_str}"

            if key not in available_keys:
                print(f"[WARN] Missing HDF5 key: {key}")
                continue

            try:
                storer = store.get_storer(key)
                nrows = getattr(storer, "nrows", None)
            except Exception:
                nrows = None

            if nrows is not None:

                for start in range(
                    0,
                    int(nrows),
                    read_chunk,
                ):

                    stop = min(
                        start + read_chunk,
                        int(nrows),
                    )

                    raw = store.select(
                        key,
                        start=start,
                        stop=stop,
                        columns=RAW_COLUMNS,
                    )

                    if len(raw) == 0:
                        continue

                    # Explicit ALL_KNOWN filtering because this reader
                    # bypasses DataZoo's dataframe filtering.
                    normalized_apps = np.array(
                        [
                            normalize_app_label(v)
                            for v in raw["APP"].to_numpy()
                        ],
                        dtype=object,
                    )

                    known_mask = np.isin(
                        normalized_apps,
                        KNOWN_APPS,
                    )

                    if not np.any(known_mask):
                        continue

                    raw = raw.loc[known_mask].copy()
                    raw["APP"] = normalized_apps[known_mask]

                    yield raw

            else:
                # Fallback for non-table HDF5 layout.
                raw = store.get(key)

                raw = raw[RAW_COLUMNS].copy()

                normalized_apps = np.array(
                    [
                        normalize_app_label(v)
                        for v in raw["APP"].to_numpy()
                    ],
                    dtype=object,
                )

                known_mask = np.isin(
                    normalized_apps,
                    KNOWN_APPS,
                )

                if not np.any(known_mask):
                    continue

                raw = raw.loc[known_mask].copy()
                raw["APP"] = normalized_apps[known_mask]

                yield raw

    finally:
        store.close()


print("HDF5 streaming reader defined.")

In [ ]:
# ============================================================
# PASS 4 — DETERMINISTIC RESERVOIR SAMPLING
# ============================================================

def reservoir_sample_period(
    dates,
    n_samples,
    seed,
    name="period",
):
    """
    Uniform reservoir sample over all known-class flows in the
    supplied dates.

    This avoids materializing millions of raw HDF5 rows at once.
    """

    rng = np.random.default_rng(seed)

    X_res = np.empty(
        (n_samples, N_FEATURES),
        dtype=np.float32,
    )

    y_res = np.empty(
        (n_samples,),
        dtype=np.int16,
    )

    filled = 0
    seen = 0

    t0 = time.time()

    for raw_chunk in iter_period_chunks(dates):

        X_chunk, y_chunk = raw_dataframe_to_numpy(raw_chunk)
        m = len(y_chunk)

        if m == 0:
            continue

        previous_seen = seen

        # ----------------------------------------------------
        # Fill the reservoir initially.
        # ----------------------------------------------------

        take = min(
            n_samples - filled,
            m,
        )

        if take > 0:

            X_res[
                filled:filled + take
            ] = X_chunk[:take]

            y_res[
                filled:filled + take
            ] = y_chunk[:take]

            filled += take

        # ----------------------------------------------------
        # Standard reservoir replacement.
        # ----------------------------------------------------

        if take < m and n_samples > 0:

            global_indices = (
                previous_seen
                + np.arange(take, m)
                + 1
            )

            probabilities = (
                n_samples / global_indices
            )

            replace_mask = (
                rng.random(m - take)
                < probabilities
            )

            chosen_local = np.flatnonzero(
                replace_mask
            )

            if len(chosen_local) > 0:

                replacement_slots = rng.integers(
                    0,
                    n_samples,
                    size=len(chosen_local),
                )

                X_res[replacement_slots] = (
                    X_chunk[take:][chosen_local]
                )

                y_res[replacement_slots] = (
                    y_chunk[take:][chosen_local]
                )

        seen += m

        del X_chunk, y_chunk, raw_chunk
        gc.collect()

        if seen % 1_000_000 < m:
            elapsed = time.time() - t0
            print(
                f"[{name}] seen={seen:,} "
                f"filled={filled:,} "
                f"elapsed={elapsed/60:.1f} min"
            )

    actual = min(filled, n_samples)

    X_res = X_res[:actual]
    y_res = y_res[:actual]

    # Deterministic final shuffle.
    final_perm = rng.permutation(actual)

    X_res = X_res[final_perm]
    y_res = y_res[final_perm]

    print(
        f"\n{name} sampling complete:"
        f"\n  rows seen    : {seen:,}"
        f"\n  rows returned: {actual:,}"
        f"\n  requested    : {n_samples:,}"
        f"\n  elapsed      : {(time.time()-t0)/60:.1f} min"
    )

    return X_res, y_res


print("Reservoir sampler ready.")
print("TEST DATA HAS NOT BEEN READ.")

In [ ]:
# ============================================================
# PASS 4 — COLLECT TRAINING SAMPLE + APRIL VALIDATION SAMPLE
# ============================================================

print("=" * 70)
print("COLLECTING TREE-BASELINE TRAINING SAMPLE")
print("=" * 70)

X_train_raw, y_train = reservoir_sample_period(
    TRAIN_DATES,
    TRAIN_BASELINE_SAMPLES,
    seed=SEED,
    name="TRAIN",
)

print("\nTraining sample shape:", X_train_raw.shape)
print("Training labels shape:", y_train.shape)

print("\nTraining class coverage:")
train_counts = np.bincount(
    y_train,
    minlength=N_CLASSES,
)

missing_train = np.flatnonzero(
    train_counts == 0
)

print(
    "Classes represented:",
    np.count_nonzero(train_counts),
    "/",
    N_CLASSES,
)

if len(missing_train):
    print(
        "[WARNING] Missing training classes:",
        [ID_TO_CLASS[int(i)] for i in missing_train],
    )
    raise RuntimeError(
        "Some classes are absent from the training sample. "
        "Increase TRAIN_BASELINE_SAMPLES."
    )

print("\n" + "=" * 70)
print("COLLECTING APRIL VALIDATION SAMPLE")
print("=" * 70)

X_val_raw, y_val = reservoir_sample_period(
    VAL_DATES,
    VAL_SAMPLES,
    seed=SEED + 1,
    name="APRIL-VAL",
)

print("\nValidation sample shape:", X_val_raw.shape)
print("Validation labels shape:", y_val.shape)

val_counts = np.bincount(
    y_val,
    minlength=N_CLASSES,
)

print(
    "Validation classes represented:",
    np.count_nonzero(val_counts),
    "/",
    N_CLASSES,
)

if np.count_nonzero(val_counts) < N_CLASSES:
    print(
        "[WARNING] Some classes absent from validation sample:"
    )
    for i in np.flatnonzero(val_counts == 0):
        print(" ", ID_TO_CLASS[int(i)])

print("\nTEST DATA HAS NOT BEEN READ.")

In [ ]:
# ============================================================
# PASS 4 — TRAIN-ONLY PADDING-SAFE PREPROCESSOR
# ============================================================

class Pass3CompatiblePreprocessor:
    """
    Preprocessing aligned with the final neural experiment:

    IPT/SIZE:
        clip at zero
        log1p
        standardize using valid non-padding positions only
        restore padding to exactly zero

    DIR:
        untouched (+1 / -1 / 0)

    Auxiliary:
        clip at zero
        log1p
        StandardScaler
        scaler fitted ONLY on training data
    """

    def __init__(self, chunk_size=250_000):
        self.chunk_size = chunk_size
        self.fitted = False

        self.ipt_mean_ = None
        self.ipt_scale_ = None

        self.size_mean_ = None
        self.size_scale_ = None

        self.aux_scaler_ = None

    def _compute_valid_channel_stats(
        self,
        X,
        value_start,
        value_end,
    ):

        total = 0
        sum_x = 0.0
        sum_x2 = 0.0

        for start in range(
            0,
            len(X),
            self.chunk_size,
        ):

            stop = min(
                start + self.chunk_size,
                len(X),
            )

            values = X[
                start:stop,
                value_start:value_end,
            ]

            directions = X[
                start:stop,
                30:60,
            ]

            valid_mask = directions != 0

            values = np.nan_to_num(
                values,
                nan=0.0,
                posinf=0.0,
                neginf=0.0,
            )

            values = np.maximum(
                values,
                0.0,
            )

            values = np.log1p(values)

            selected = values[valid_mask]

            if selected.size == 0:
                continue

            selected64 = selected.astype(
                np.float64,
                copy=False,
            )

            total += selected64.size
            sum_x += float(
                selected64.sum()
            )
            sum_x2 += float(
                np.square(selected64).sum()
            )

        if total == 0:
            raise RuntimeError(
                "No valid non-padding values found."
            )

        mean = sum_x / total

        variance = (
            sum_x2 / total
            - mean * mean
        )

        variance = max(
            variance,
            1e-12,
        )

        std = math.sqrt(variance)

        return float(mean), float(std)

    def fit(self, X):

        print("Fitting IPT statistics...")
        self.ipt_mean_, self.ipt_scale_ = (
            self._compute_valid_channel_stats(
                X,
                0,
                30,
            )
        )

        print(
            "  IPT mean:",
            self.ipt_mean_,
            "scale:",
            self.ipt_scale_,
        )

        print("Fitting SIZE statistics...")
        self.size_mean_, self.size_scale_ = (
            self._compute_valid_channel_stats(
                X,
                60,
                90,
            )
        )

        print(
            "  SIZE mean:",
            self.size_mean_,
            "scale:",
            self.size_scale_,
        )

        # ----------------------------------------------------
        # Auxiliary scaler — train only.
        # ----------------------------------------------------

        print("Fitting auxiliary StandardScaler...")

        self.aux_scaler_ = StandardScaler()

        for start in range(
            0,
            len(X),
            self.chunk_size,
        ):

            stop = min(
                start + self.chunk_size,
                len(X),
            )

            aux = X[
                start:stop,
                90:101,
            ]

            aux = np.nan_to_num(
                aux,
                nan=0.0,
                posinf=0.0,
                neginf=0.0,
            )

            aux = np.maximum(
                aux,
                0.0,
            )

            aux = np.log1p(aux)

            self.aux_scaler_.partial_fit(
                aux
            )

        self.fitted = True

        print("Preprocessor fitted on TRAIN ONLY.")

        return self

    def transform_inplace(self, X):

        if not self.fitted:
            raise RuntimeError(
                "Preprocessor must be fitted first."
            )

        np.nan_to_num(
            X,
            copy=False,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        for start in range(
            0,
            len(X),
            self.chunk_size,
        ):

            stop = min(
                start + self.chunk_size,
                len(X),
            )

            directions = X[
                start:stop,
                30:60,
            ]

            valid_mask = directions != 0

            # ------------------------------------------------
            # IPT
            # ------------------------------------------------

            ipt = X[
                start:stop,
                0:30,
            ]

            np.maximum(
                ipt,
                0.0,
                out=ipt,
            )

            np.log1p(
                ipt,
                out=ipt,
            )

            ipt -= self.ipt_mean_
            ipt /= self.ipt_scale_

            # Padding is restored to exact zero.
            ipt[~valid_mask] = 0.0

            # ------------------------------------------------
            # SIZE
            # ------------------------------------------------

            size = X[
                start:stop,
                60:90,
            ]

            np.maximum(
                size,
                0.0,
                out=size,
            )

            np.log1p(
                size,
                out=size,
            )

            size -= self.size_mean_
            size /= self.size_scale_

            size[~valid_mask] = 0.0

            # ------------------------------------------------
            # DIR is deliberately untouched.
            # ------------------------------------------------

            # ------------------------------------------------
            # AUXILIARY FEATURES
            # ------------------------------------------------

            aux = X[
                start:stop,
                90:101,
            ]

            np.maximum(
                aux,
                0.0,
                out=aux,
            )

            np.log1p(
                aux,
                out=aux,
            )

            aux -= self.aux_scaler_.mean_.astype(
                np.float32
            )

            aux /= self.aux_scaler_.scale_.astype(
                np.float32
            )

        return X

    def transform(self, X):

        X_copy = np.array(
            X,
            dtype=np.float32,
            copy=True,
        )

        return self.transform_inplace(
            X_copy
        )


print("Padding-safe preprocessing class defined.")

In [ ]:
# ============================================================
# PASS 4 — FIT PREPROCESSOR ON TRAIN ONLY + SAVE ARRAYS
# ============================================================

t0 = time.time()

preprocessor = Pass3CompatiblePreprocessor(
    chunk_size=250_000
)

preprocessor.fit(
    X_train_raw
)

print("\nTransforming training sample...")
preprocessor.transform_inplace(
    X_train_raw
)

print("Transforming April validation sample...")
preprocessor.transform_inplace(
    X_val_raw
)

print(
    "\nPreprocessing elapsed:",
    f"{(time.time()-t0)/60:.2f} minutes"
)

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert np.isfinite(X_train_raw).all()
assert np.isfinite(X_val_raw).all()

print("\nFinite-value checks: PASSED")

print(
    "Train shape:",
    X_train_raw.shape
)

print(
    "Val shape  :",
    X_val_raw.shape
)

print(
    "Train dtype:",
    X_train_raw.dtype
)

print(
    "Val dtype  :",
    X_val_raw.dtype
)

# Direction should remain discrete.
unique_dir = np.unique(
    X_train_raw[:, 30:60]
)

print(
    "\nUnique direction values:",
    unique_dir[:20]
)

# Save transformed arrays.
np.save(
    RESULT_DIR / "X_train.npy",
    X_train_raw,
)

np.save(
    RESULT_DIR / "y_train.npy",
    y_train,
)

np.save(
    RESULT_DIR / "X_val.npy",
    X_val_raw,
)

np.save(
    RESULT_DIR / "y_val.npy",
    y_val,
)

# Save preprocessing object.
joblib.dump(
    preprocessor,
    RESULT_DIR / "preprocessor.joblib",
)

# Save label mapping.
with open(
    RESULT_DIR / "label_mapping.json",
    "w",
) as f:
    json.dump(
        {
            "class_to_id": CLASS_TO_ID,
            "id_to_class": {
                str(k): v
                for k, v in ID_TO_CLASS.items()
            },
        },
        f,
        indent=2,
    )

print("\nSaved:")
print(
    RESULT_DIR / "X_train.npy"
)
print(
    RESULT_DIR / "X_val.npy"
)
print(
    RESULT_DIR / "preprocessor.joblib"
)

# Release raw training arrays after serialization.
del X_train_raw
del X_val_raw

gc.collect()

print("\nMemory cleanup complete.")
print("TEST DATA HAS NOT BEEN READ.")

In [ ]:
# ============================================================
# PASS 4 — LOAD TRAIN/VALIDATION MATRICES + METRIC FUNCTIONS
# ============================================================

X_train = np.load(
    RESULT_DIR / "X_train.npy",
    mmap_mode="r",
)

y_train = np.load(
    RESULT_DIR / "y_train.npy",
)

X_val = np.load(
    RESULT_DIR / "X_val.npy",
    mmap_mode="r",
)

y_val = np.load(
    RESULT_DIR / "y_val.npy",
)

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)

print("X_val  :", X_val.shape, X_val.dtype)
print("y_val  :", y_val.shape, y_val.dtype)


def expected_calibration_error(
    y_true,
    probabilities,
    n_bins=10,
):
    """
    Standard top-confidence ECE.
    """

    probabilities = np.asarray(
        probabilities,
        dtype=np.float32,
    )

    y_true = np.asarray(
        y_true
    )

    predicted = probabilities.argmax(
        axis=1
    )

    confidence = probabilities.max(
        axis=1
    )

    correctness = (
        predicted == y_true
    ).astype(
        np.float32
    )

    bin_edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1,
    )

    ece = 0.0

    for i in range(n_bins):

        left = bin_edges[i]
        right = bin_edges[i + 1]

        if i == n_bins - 1:
            mask = (
                (confidence >= left)
                & (confidence <= right)
            )
        else:
            mask = (
                (confidence >= left)
                & (confidence < right)
            )

        if not np.any(mask):
            continue

        bin_accuracy = correctness[
            mask
        ].mean()

        bin_confidence = confidence[
            mask
        ].mean()

        bin_fraction = (
            mask.mean()
        )

        ece += (
            bin_fraction
            * abs(
                bin_accuracy
                - bin_confidence
            )
        )

    return float(ece)


def calculate_metrics(
    y_true,
    probabilities,
):
    """
    Calculate the metrics used in the paper.
    """

    probabilities = np.asarray(
        probabilities,
        dtype=np.float32,
    )

    predictions = probabilities.argmax(
        axis=1
    )

    metrics = {
        "accuracy": float(
            accuracy_score(
                y_true,
                predictions,
            )
        ),
        "macro_precision": float(
            precision_score(
                y_true,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_recall": float(
            recall_score(
                y_true,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_true,
                predictions,
                average="weighted",
                zero_division=0,
            )
        ),
        "ece_10": expected_calibration_error(
            y_true,
            probabilities,
            n_bins=10,
        ),
    }

    return metrics, predictions


def save_evaluation(
    model_name,
    split_name,
    y_true,
    probabilities,
):
    """
    Save probabilities + per-class report + aggregate metrics.
    """

    probabilities = np.asarray(
        probabilities,
        dtype=np.float32,
    )

    metrics, predictions = calculate_metrics(
        y_true,
        probabilities,
    )

    metrics["model"] = model_name
    metrics["split"] = split_name
    metrics["samples"] = int(len(y_true))

    # Save probability tensor.
    np.save(
        RESULT_DIR
        / f"{model_name}_{split_name}_probs.npy",
        probabilities,
    )

    # Save predictions.
    np.save(
        RESULT_DIR
        / f"{model_name}_{split_name}_pred.npy",
        predictions,
    )

    # Save classification report.
    report = classification_report(
        y_true,
        predictions,
        labels=np.arange(N_CLASSES),
        target_names=KNOWN_APPS,
        output_dict=True,
        zero_division=0,
    )

    report_df = pd.DataFrame(
        report
    ).T

    report_df.to_csv(
        RESULT_DIR
        / f"{model_name}_{split_name}_classification_report.csv"
    )

    print(
        f"\n{model_name} — {split_name}"
    )
    print(
        f"  Accuracy       : {metrics['accuracy']:.6f}"
    )
    print(
        f"  Macro precision: {metrics['macro_precision']:.6f}"
    )
    print(
        f"  Macro recall   : {metrics['macro_recall']:.6f}"
    )
    print(
        f"  Macro F1       : {metrics['macro_f1']:.6f}"
    )
    print(
        f"  Weighted F1    : {metrics['weighted_f1']:.6f}"
    )
    print(
        f"  ECE (10-bin)   : {metrics['ece_10']:.6f}"
    )

    return metrics


print("\nMetric functions ready.")

In [ ]:
# ============================================================
# PASS 4 — TRAIN LIGHTGBM BASELINE
# ============================================================

print("=" * 70)
print("TRAINING LIGHTGBM BASELINE")
print("=" * 70)

lgbm_config = {
    "objective": "multiclass",
    "num_class": N_CLASSES,

    # Fixed, non-HPO baseline configuration.
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 100,

    "subsample": 0.80,
    "subsample_freq": 1,
    "colsample_bytree": 0.80,

    "reg_alpha": 0.0,
    "reg_lambda": 1.0,

    "random_state": SEED,

    # Use CPU for maximum compatibility.
    "device_type": "cpu",
    "n_jobs": max(
        1,
        (os.cpu_count() or 2) - 1,
    ),

    "verbosity": -1,
}

print("Configuration:")
print(
    json.dumps(
        lgbm_config,
        indent=2,
    )
)

lgbm = LGBMClassifier(
    **lgbm_config
)

t0 = time.time()

lgbm.fit(
    X_train,
    y_train,
    eval_set=[
        (X_val, y_val),
    ],
    eval_metric="multi_logloss",
    callbacks=[
        # This is only model fitting/early stopping;
        # the final metric comparison remains macro-F1.
        __import__("lightgbm").early_stopping(
            stopping_rounds=50,
            verbose=True,
        ),
        __import__("lightgbm").log_evaluation(
            period=50
        ),
    ],
)

elapsed = time.time() - t0

print(
    "\nLightGBM training time:",
    f"{elapsed/3600:.2f} hours"
)

print(
    "Best iteration:",
    getattr(
        lgbm,
        "best_iteration_",
        None,
    )
)

# ------------------------------------------------------------
# Save model
# ------------------------------------------------------------

lgbm_path = (
    RESULT_DIR
    / "lightgbm_baseline.joblib"
)

joblib.dump(
    lgbm,
    lgbm_path,
)

with open(
    RESULT_DIR / "lightgbm_config.json",
    "w",
) as f:
    json.dump(
        {
            "config": lgbm_config,
            "best_iteration": getattr(
                lgbm,
                "best_iteration_",
                None,
            ),
        },
        f,
        indent=2,
    )

print(
    "Saved:",
    lgbm_path
)

# ------------------------------------------------------------
# Validation evaluation
# ------------------------------------------------------------

print(
    "\nGenerating LightGBM validation probabilities..."
)

lgbm_val_probs = lgbm.predict_proba(
    X_val
).astype(
    np.float32
)

lgbm_val_metrics = save_evaluation(
    "lightgbm",
    "val",
    y_val,
    lgbm_val_probs,
)

with open(
    RESULT_DIR / "lightgbm_val_metrics.json",
    "w",
) as f:
    json.dump(
        lgbm_val_metrics,
        f,
        indent=2,
    )

del lgbm_val_probs
del lgbm
gc.collect()

print("\nLightGBM baseline complete.")
print("TEST DATA HAS NOT BEEN READ.")

In [ ]:
# ============================================================
# PASS 4 — TRAIN XGBOOST BASELINE
# ============================================================

print("=" * 70)
print("TRAINING XGBOOST BASELINE")
print("=" * 70)

# ------------------------------------------------------------
# Automatically select CUDA when NVIDIA is available.
# ------------------------------------------------------------

XGB_DEVICE = (
    "cuda"
    if GPU_VISIBLE
    else "cpu"
)

xgb_config = {
    "objective": "multi:softprob",
    "num_class": N_CLASSES,

    # Fixed, non-HPO baseline configuration.
    "n_estimators": 500,
    "learning_rate": 0.05,
    "max_depth": 8,
    "min_child_weight": 5,

    "subsample": 0.80,
    "colsample_bytree": 0.80,

    "reg_alpha": 0.0,
    "reg_lambda": 1.0,

    "tree_method": "hist",
    "device": XGB_DEVICE,

    "eval_metric": "mlogloss",

    "early_stopping_rounds": 50,

    "random_state": SEED,
    "n_jobs": max(
        1,
        (os.cpu_count() or 2) - 1,
    ),

    "verbosity": 1,
}

print("XGBoost device:", XGB_DEVICE)

print(
    "\nConfiguration:"
)
print(
    json.dumps(
        xgb_config,
        indent=2,
    )
)

xgb = XGBClassifier(
    **xgb_config
)

t0 = time.time()

try:

    xgb.fit(
        X_train,
        y_train,
        eval_set=[
            (
                X_val,
                y_val,
            )
        ],
        verbose=50,
    )

except Exception as exc:

    print(
        "\nXGBoost CUDA training failed."
    )
    print(
        "Error:",
        repr(exc)
    )

    if XGB_DEVICE != "cuda":
        raise

    print(
        "\nRetrying XGBoost on CPU..."
    )

    del xgb
    gc.collect()

    xgb_config["device"] = "cpu"

    xgb = XGBClassifier(
        **xgb_config
    )

    xgb.fit(
        X_train,
        y_train,
        eval_set=[
            (
                X_val,
                y_val,
            )
        ],
        verbose=50,
    )

    XGB_DEVICE = "cpu"

elapsed = time.time() - t0

print(
    "\nXGBoost training time:",
    f"{elapsed/3600:.2f} hours"
)

print(
    "Best iteration:",
    getattr(
        xgb,
        "best_iteration",
        None,
    )
)

# ------------------------------------------------------------
# Save model
# ------------------------------------------------------------

xgb_path = (
    RESULT_DIR
    / "xgboost_baseline.joblib"
)

joblib.dump(
    xgb,
    xgb_path,
)

with open(
    RESULT_DIR / "xgboost_config.json",
    "w",
) as f:
    json.dump(
        {
            "config": xgb_config,
            "actual_device": XGB_DEVICE,
            "best_iteration": getattr(
                xgb,
                "best_iteration",
                None,
            ),
        },
        f,
        indent=2,
    )

print(
    "Saved:",
    xgb_path
)

# ------------------------------------------------------------
# Validation evaluation
# ------------------------------------------------------------

print(
    "\nGenerating XGBoost validation probabilities..."
)

xgb_val_probs = xgb.predict_proba(
    X_val
).astype(
    np.float32
)

xgb_val_metrics = save_evaluation(
    "xgboost",
    "val",
    y_val,
    xgb_val_probs,
)

with open(
    RESULT_DIR / "xgboost_val_metrics.json",
    "w",
) as f:
    json.dump(
        xgb_val_metrics,
        f,
        indent=2,
    )

del xgb_val_probs
del xgb
gc.collect()

print("\nXGBoost baseline complete.")
print("TEST DATA HAS NOT BEEN READ.")

In [ ]:
# ============================================================
# PASS 4 — ONLY NOW READ MAY 2025 TEST DATA
# ============================================================

print("=" * 70)
print("FINAL TEST DATA ACCESS")
print("=" * 70)

print(
    "The two tree baselines have already been trained "
    "and validation-evaluated."
)

print(
    "Reading May 2025 test data NOW."
)

t0 = time.time()

X_test_raw_parts = []
y_test_parts = []

rows_seen = 0

for raw_chunk in iter_period_chunks(
    TEST_DATES
):

    X_chunk, y_chunk = raw_dataframe_to_numpy(
        raw_chunk
    )

    if len(y_chunk) == 0:
        continue

    X_test_raw_parts.append(
        X_chunk
    )

    y_test_parts.append(
        y_chunk
    )

    rows_seen += len(y_chunk)

    if rows_seen % 500_000 < len(y_chunk):
        print(
            f"Test rows collected: "
            f"{rows_seen:,}"
        )

    del X_chunk, y_chunk, raw_chunk
    gc.collect()

print(
    "\nRaw test rows:",
    rows_seen,
)

X_test_raw = np.concatenate(
    X_test_raw_parts,
    axis=0,
)

y_test = np.concatenate(
    y_test_parts,
    axis=0,
)

del X_test_raw_parts
del y_test_parts

gc.collect()

print(
    "Raw test shape:",
    X_test_raw.shape
)

print(
    "Raw test labels:",
    y_test.shape
)

# ------------------------------------------------------------
# Confirm expected May test size.
# ------------------------------------------------------------

assert len(y_test) == 2_359_061, (
    f"Expected 2,359,061 test flows, "
    f"found {len(y_test):,}"
)

# ------------------------------------------------------------
# Transform with TRAIN-FITTED preprocessor.
# ------------------------------------------------------------

print(
    "\nApplying TRAIN-FITTED preprocessing "
    "to May test..."
)

preprocessor = joblib.load(
    RESULT_DIR / "preprocessor.joblib"
)

preprocessor.transform_inplace(
    X_test_raw
)

X_test = X_test_raw

assert np.isfinite(
    X_test
).all()

print(
    "Test preprocessing complete."
)

print(
    "Elapsed:",
    f"{(time.time()-t0)/60:.2f} minutes"
)

# ------------------------------------------------------------
# Save test arrays.
# ------------------------------------------------------------

np.save(
    RESULT_DIR / "X_test.npy",
    X_test,
)

np.save(
    RESULT_DIR / "y_test.npy",
    y_test,
)

print(
    "\nSaved final test arrays."
)

del X_test_raw
gc.collect()

print(
    "X_test now remains in memory for final evaluation."
)

In [ ]:
# ============================================================
# PASS 4 — FINAL MAY TEST: LIGHTGBM
# ============================================================

print("=" * 70)
print("LIGHTGBM — FINAL MAY 2025 TEST")
print("=" * 70)

lgbm = joblib.load(
    RESULT_DIR / "lightgbm_baseline.joblib"
)

t0 = time.time()

lgbm_test_probs = lgbm.predict_proba(
    X_test
).astype(
    np.float32
)

print(
    "Prediction time:",
    f"{time.time()-t0:.2f} sec"
)

lgbm_test_metrics = save_evaluation(
    "lightgbm",
    "test",
    y_test,
    lgbm_test_probs,
)

with open(
    RESULT_DIR / "lightgbm_test_metrics.json",
    "w",
) as f:
    json.dump(
        lgbm_test_metrics,
        f,
        indent=2,
    )

del lgbm_test_probs
del lgbm

gc.collect()

print(
    "\nLightGBM FINAL TEST COMPLETE."
)

In [ ]:
# ============================================================
# PASS 4 — FINAL MAY TEST: XGBOOST
# ============================================================

print("=" * 70)
print("XGBOOST — FINAL MAY 2025 TEST")
print("=" * 70)

xgb = joblib.load(
    RESULT_DIR / "xgboost_baseline.joblib"
)

t0 = time.time()

xgb_test_probs = xgb.predict_proba(
    X_test
).astype(
    np.float32
)

print(
    "Prediction time:",
    f"{time.time()-t0:.2f} sec"
)

xgb_test_metrics = save_evaluation(
    "xgboost",
    "test",
    y_test,
    xgb_test_probs,
)

with open(
    RESULT_DIR / "xgboost_test_metrics.json",
    "w",
) as f:
    json.dump(
        xgb_test_metrics,
        f,
        indent=2,
    )

del xgb_test_probs
del xgb

gc.collect()

print(
    "\nXGBoost FINAL TEST COMPLETE."
)

In [ ]:
# ============================================================
# PASS 4 — MAJORITY SANITY BASELINE + FINAL COMPARISON TABLE
# ============================================================

# ------------------------------------------------------------
# Majority classifier
# ------------------------------------------------------------

majority_class = int(
    np.bincount(
        y_train,
        minlength=N_CLASSES,
    ).argmax()
)

majority_probabilities = np.zeros(
    (len(y_test), N_CLASSES),
    dtype=np.float32,
)

majority_probabilities[
    :,
    majority_class,
] = 1.0

majority_metrics = save_evaluation(
    "majority",
    "test",
    y_test,
    majority_probabilities,
)

del majority_probabilities
gc.collect()

print(
    "\nMajority class:",
    ID_TO_CLASS[majority_class],
)

# ------------------------------------------------------------
# Load computed baseline metrics.
# ------------------------------------------------------------

with open(
    RESULT_DIR / "lightgbm_test_metrics.json"
) as f:
    lightgbm_test_metrics = json.load(f)

with open(
    RESULT_DIR / "xgboost_test_metrics.json"
) as f:
    xgboost_test_metrics = json.load(f)

# ------------------------------------------------------------
# Latest Pass 3 neural result.
#
# IMPORTANT:
# These values are copied from the latest completed Pass 3 run,
# NOT recalculated in Pass 4.
# ------------------------------------------------------------

neural_pass3 = {
    "model": "Pass3 A+C stacking",
    "split": "test",
    "samples": 2_359_061,

    "accuracy": 0.9324,
    "macro_precision": 0.8725,
    "macro_recall": 0.8272,
    "macro_f1": 0.8386,
    "weighted_f1": 0.9320,
    "ece_10": 0.0096,
}

# ------------------------------------------------------------
# Build comparison table.
# ------------------------------------------------------------

comparison_rows = [
    neural_pass3,

    {
        "model": "LightGBM",
        "split": "test",
        "samples": lightgbm_test_metrics["samples"],
        "accuracy": lightgbm_test_metrics["accuracy"],
        "macro_precision": lightgbm_test_metrics["macro_precision"],
        "macro_recall": lightgbm_test_metrics["macro_recall"],
        "macro_f1": lightgbm_test_metrics["macro_f1"],
        "weighted_f1": lightgbm_test_metrics["weighted_f1"],
        "ece_10": lightgbm_test_metrics["ece_10"],
    },

    {
        "model": "XGBoost",
        "split": "test",
        "samples": xgboost_test_metrics["samples"],
        "accuracy": xgboost_test_metrics["accuracy"],
        "macro_precision": xgboost_test_metrics["macro_precision"],
        "macro_recall": xgboost_test_metrics["macro_recall"],
        "macro_f1": xgboost_test_metrics["macro_f1"],
        "weighted_f1": xgboost_test_metrics["weighted_f1"],
        "ece_10": xgboost_test_metrics["ece_10"],
    },

    {
        "model": "Majority",
        "split": "test",
        "samples": majority_metrics["samples"],
        "accuracy": majority_metrics["accuracy"],
        "macro_precision": majority_metrics["macro_precision"],
        "macro_recall": majority_metrics["macro_recall"],
        "macro_f1": majority_metrics["macro_f1"],
        "weighted_f1": majority_metrics["weighted_f1"],
        "ece_10": majority_metrics["ece_10"],
    },
]

comparison_df = pd.DataFrame(
    comparison_rows
)

comparison_df = comparison_df[
    [
        "model",
        "samples",
        "accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "weighted_f1",
        "ece_10",
    ]
]

comparison_df = comparison_df.sort_values(
    "macro_f1",
    ascending=False,
).reset_index(
    drop=True
)

print(
    "\n"
    + "=" * 100
)

print(
    "PASS 4 FINAL COMPARISON"
)

print(
    "=" * 100
)

display(
    comparison_df.style.format(
        {
            "accuracy": "{:.4f}",
            "macro_precision": "{:.4f}",
            "macro_recall": "{:.4f}",
            "macro_f1": "{:.4f}",
            "weighted_f1": "{:.4f}",
            "ece_10": "{:.4f}",
        }
    )
)

comparison_df.to_csv(
    RESULT_DIR
    / "Pass4_comparison.csv",
    index=False,
)

with open(
    RESULT_DIR / "Pass4_comparison.json",
    "w",
) as f:
    json.dump(
        comparison_rows,
        f,
        indent=2,
    )

print(
    "\nSaved:",
    RESULT_DIR / "Pass4_comparison.csv"
)

In [ ]:
# ============================================================
# PASS 4 — VALIDATION COMPARISON
# ============================================================

validation_rows = []

with open(
    RESULT_DIR / "lightgbm_val_metrics.json"
) as f:
    lgbm_val = json.load(f)

with open(
    RESULT_DIR / "xgboost_val_metrics.json"
) as f:
    xgb_val = json.load(f)

validation_rows.append(
    {
        "model": "LightGBM",
        "samples": lgbm_val["samples"],
        "accuracy": lgbm_val["accuracy"],
        "macro_precision": lgbm_val["macro_precision"],
        "macro_recall": lgbm_val["macro_recall"],
        "macro_f1": lgbm_val["macro_f1"],
        "weighted_f1": lgbm_val["weighted_f1"],
        "ece_10": lgbm_val["ece_10"],
    }
)

validation_rows.append(
    {
        "model": "XGBoost",
        "samples": xgb_val["samples"],
        "accuracy": xgb_val["accuracy"],
        "macro_precision": xgb_val["macro_precision"],
        "macro_recall": xgb_val["macro_recall"],
        "macro_f1": xgb_val["macro_f1"],
        "weighted_f1": xgb_val["weighted_f1"],
        "ece_10": xgb_val["ece_10"],
    }
)

validation_df = pd.DataFrame(
    validation_rows
)

display(
    validation_df.style.format(
        {
            "accuracy": "{:.4f}",
            "macro_precision": "{:.4f}",
            "macro_recall": "{:.4f}",
            "macro_f1": "{:.4f}",
            "weighted_f1": "{:.4f}",
            "ece_10": "{:.4f}",
        }
    )
)

validation_df.to_csv(
    RESULT_DIR
    / "Pass4_validation_comparison.csv",
    index=False,
)

print(
    "\nSaved:",
    RESULT_DIR
    / "Pass4_validation_comparison.csv"
)

In [ ]:
# ============================================================
# PASS 4 — FINAL EXPERIMENT MANIFEST
# ============================================================

FINAL_MANIFEST = {
    "pass": 4,
    "dataset": "CESNET-QUICEXT-25",
    "dataset_size": DATASET_SIZE,

    "label_policy": "ALL_KNOWN",
    "n_classes": N_CLASSES,
    "classes": KNOWN_APPS,

    "train_period": {
        "start": "2024-06-01",
        "end": "2025-03-31",
        "baseline_sample": TRAIN_BASELINE_SAMPLES,
        "sampling": "uniform reservoir sampling",
        "seed": SEED,
    },

    "validation_period": {
        "start": "2025-04-01",
        "end": "2025-04-30",
        "samples": VAL_SAMPLES,
        "sampling": "uniform reservoir sampling",
        "seed": SEED + 1,
    },

    "test_period": {
        "start": "2025-05-01",
        "end": "2025-05-31",
        "samples": int(len(y_test)),
        "sampling": "all known-class flows",
    },

    "features": {
        "ppi": [
            "IPT",
            "DIR",
            "SIZE",
        ],
        "ppi_packets": 30,
        "auxiliary": AUX_FEATURES,
        "total_dimensions": N_FEATURES,
    },

    "preprocessing": {
        "ipt": "clip -> log1p -> train-only valid-position standardization",
        "dir": "unchanged",
        "size": "clip -> log1p -> train-only valid-position standardization",
        "auxiliary": "clip -> log1p -> train-only StandardScaler",
        "padding": "restored to zero after IPT/SIZE normalization",
    },

    "models": {
        "LightGBM": "fixed configuration, no HPO",
        "XGBoost": "fixed configuration, no HPO",
        "majority": "sanity baseline",
    },

    "neural_reference": {
        "model": "Pass3 A+C stacking",
        "accuracy": 0.9324,
        "macro_f1": 0.8386,
        "weighted_f1": 0.9320,
        "ece_10": 0.0096,
    },

    "versions": VERSIONS,

    "results": {
        "comparison_csv": str(
            RESULT_DIR / "Pass4_comparison.csv"
        ),
        "validation_csv": str(
            RESULT_DIR / "Pass4_validation_comparison.csv"
        ),
    },
}

with open(
    RESULT_DIR / "Pass4_final_manifest.json",
    "w",
) as f:
    json.dump(
        FINAL_MANIFEST,
        f,
        indent=2,
    )

print(
    json.dumps(
        {
            "status": "PASS 4 COMPLETE",
            "result_dir": str(RESULT_DIR),
            "comparison": str(
                RESULT_DIR
                / "Pass4_comparison.csv"
            ),
            "manifest": str(
                RESULT_DIR
                / "Pass4_final_manifest.json"
            ),
        },
        indent=2,
    )
)